Based on:
https://github.com/ErwannMillon/Color-diffusion

# Libraries

In [3]:
pip install pytorch-lightning

In [4]:
pip install einops

In [5]:
%matplotlib inline
# %config InlineBackend.figure_format = 'retina'

from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [5, 5]
from glob import glob
import os
from copy import deepcopy

import numpy as np
from skimage import util
import cv2
from PIL import Image

import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision.transforms import v2, functional
from torchvision import transforms
from sklearn.model_selection import train_test_split

from urllib.request import urlretrieve
import pandas as pd
import requests
from io import BytesIO
import random


In [10]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import requests
from PIL import Image
from io import BytesIO
from skimage import util
from sklearn.model_selection import train_test_split

# Utils

In [3]:
from glob import glob
import numpy as np
from skimage.color import lab2rgb
import matplotlib.pyplot as plt
from PIL import Image
from torch import nn
import torch

In [4]:
def get_device():
    try:
        if torch.backends.mps.is_available() and torch.backends.mps.is_built():
            return "mps"
    except:
        device = "cpu"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    return (device)

In [5]:

def lab_to_pil(img):
    if len(img.shape) == 3:
        img = img.unsqueeze(0)
    rgb_img = lab_to_rgb(*split_lab_channels(img))
    pil_img = Image.fromarray(np.uint8(rgb_img[0] * 255))
    return pil_img


def freeze_module(module):
    for param in module.parameters():
        param.requires_grad = False

def custom_to_pil(x, process=True):
    x = x.detach().cpu()
    if process:
        x = torch.clamp(x, -1., 1.)
        x = (x + 1.)/2.
    x = x.permute(1, 2, 0).numpy()
    if process:
        x = (255*x).astype(np.uint8)
    return x


def show_lab_image(image, stepsize=10, log=True, caption="diff samples"):
    plt.figure(figsize=(20, 9))
    rgb_imgs = lab_to_rgb(*split_lab_channels(image))
    plt.imshow(rgb_imgs[0])
    plt.show()


def init_weights(net, init='norm', gain=2**0.5, leakyslope=0.02):
    def init_func(m):
        classname = m.__class__.__name__
        if hasattr(m, 'weight') and 'Conv' in classname:
            if init == 'norm':
                nn.init.normal_(m.weight.data, mean=0.0, std=gain)
            elif init == 'xavier':
                nn.init.xavier_normal_(m.weight.data, gain=gain)
            elif init == 'kaiming':
                nn.init.kaiming_normal_(m.weight.data, mode='fan_in', nonlinearity='relu')
            if hasattr(m, 'bias') and m.bias is not None:
                nn.init.constant_(m.bias.data, 0.0)
        elif 'BatchNorm2d' in classname:
            nn.init.normal_(m.weight.data, 1., gain)
            nn.init.constant_(m.bias.data, 0.)
    net.apply(init_func)
    print(f"model initialized with {init} initialization")
    return net


def init_model(model, device, init):
    model = init_weights(model, init)
    return model


def right_pad_dims_to(x: torch.tensor, t: torch.tensor) -> torch.tensor:
    """
    Pads `t` with empty dimensions to the number of dimensions `x` has. If `t` does not have fewer dimensions than `x`
        it is returned without change.
    """
    padding_dims = x.ndim - t.ndim
    if padding_dims <= 0:
        return t
    return t.view(*t.shape, *((1,) * padding_dims))


def split_lab_channels(image):
    assert isinstance(image, torch.Tensor)
    if len(image.shape) == 3:
        image = image.unsqueeze(0)
    return torch.split(image, [1, 2], dim=1)


def cat_lab(L, ab):
    return (torch.cat((L, ab), dim=1))


def lab_to_rgb(L, ab):
    """
    Converts a batch of torch tensors from Lab to RGB
    """
    L = (L + 1.) * 50.
    ab = ab * 110.
    Lab = torch.cat([L, ab], dim=1).permute(0, 2, 3, 1).cpu().numpy()
    rgb_imgs = []
    for img in Lab:
        img_rgb = lab2rgb(img)
        rgb_imgs.append(img_rgb)
    return np.stack(rgb_imgs, axis=0)


def l_to_rgb(L):
    """Converts a single channel greyscale image to RGB"""
    if len(L.shape) == 3:
        L = L.unsqueeze(0)
    L = (L + 1.) * 50.
    print(L.min(), L.max())
    return L.repeat(3, dim=1)

# Define helper functions

In [6]:
## convert RGB to the personal LAB (LAB2)
# the input R,G,B,  must be 1D from 0 to 255
# the outputs are 1D  L [0 1], a [-1 1] b [-1 1]
def RGB2LAB2(R0, G0, B0):

    R=R0/255
    G=G0/255
    B=B0/255

    # Y=0.3*R + 0.59*G + 0.11*B
    # X=0.45*R + 0.35*G + 0.2*B
    # Z=0.01*R + 0.09*G + 0.9*B

    Y=0.299*R + 0.587*G + 0.114*B
    X=0.449*R + 0.353*G + 0.198*B
    Z=0.012*R + 0.089*G + 0.899*B

    # X - Y = 0.150*R - 0.234*G + 0.084*B  = a0
    # Y - Z = 0.287*R + 0.498*G - 0.785*B  = b0

    L = Y
    a = (X - Y)/0.234
    b = (Y - Z)/0.785

    return L, a, b

## convert the personal LAB (LAB2)to the RGB
# the input L,a,b,  must be 1D L [0 1], a [-1 1] b [-1 1]
# the outputs are 1D  R g B [0 255]
def LAB22RGB(L, a, b):

    a11 = 0.299
    a12 = 0.587
    a13 = 0.114
    a21 = (0.15/0.234)
    a22 = (-0.234/0.234)
    a23 = (0.084/0.234)
    a31 = (0.287/0.785)
    a32 = (0.498/0.785)
    a33 = (-0.785/0.785)

    aa=np.array([[a11, a12, a13], [a21, a22, a23], [a31, a32, a33]])
    C0=np.zeros((L.shape[0],3))
    C0[:,0]=L[:,0]
    C0[:,1]=a[:,0]
    C0[:,2]=b[:,0]
    C = np.transpose(C0)
    # C = np.array([L, a, b])
    # print(C.shape)
    # print(L.shape)
    # print(a.shape)
    # print(b.shape)
    # print(aa.shape)

    X = np.linalg.inv(aa).dot(C)
    X1D=np.reshape(X,(X.shape[0]*X.shape[1],1))
    p0=np.where(X1D<0)
    X1D[p0[0]]=0
    p1=np.where(X1D>1)
    X1D[p1[0]]=1
    Xr=np.reshape(X1D,(X.shape[0],X.shape[1]))

    Rr = Xr[0][:]
    Gr = Xr[1][:]
    Br = Xr[2][:]

    R = np.uint(np.round(Rr*255))
    G = np.uint(np.round(Gr*255))
    B = np.uint(np.round(Br*255))
    # p0=np.where(L<0.02)
    # R[p0[0]]=0
    # G[p0[0]]=0
    # B[p0[0]]=0
    # p1=np.where(L>0.98)
    # R[p1[0]]=255
    # G[p1[0]]=255
    # B[p1[0]]=255
    return R, G, B


def convert_RGB_to_feed_model(img):
    img = np.asarray(img)
    sz_x = img.shape[0]
    sz_y = img.shape[1]

    train_imgs = np.zeros((sz_x, sz_y, 2))
    train_input = np.zeros((sz_x, sz_y, 1))

    R1 = np.reshape(img[:, :, 0], (sz_x * sz_y, 1))
    G1 = np.reshape(img[:, :, 1], (sz_x * sz_y, 1))
    B1 = np.reshape(img[:, :, 2], (sz_x * sz_y, 1))
    L, A, B = RGB2LAB2(R1, G1, B1)

    train_input[:, :, 0] = L.reshape((sz_x, sz_y))
    train_imgs[:, :, 0] = np.reshape(A, (sz_x, sz_y))
    train_imgs[:, :, 1] = np.reshape(B, (sz_x, sz_y))

    return train_input, train_imgs

def convert_to_LAB_transform(image):
    L, AB = convert_RGB_to_feed_model(image)
    return (L, AB)

# Define custom Dataset class


In [7]:
class SwisstopoDataset:
    def __init__(self, img_indx, transform=None, large_dataset=False, return_label=True, batch_size=32, shuffle=False):
        self.img_indx = img_indx
        self.transform = transform
        self.large_dataset = large_dataset
        self.return_label = return_label
        self.batch_size = batch_size
        self.shuffle = shuffle

        # Set the appropriate port based on the dataset size
        self.port = 1986 if self.large_dataset else 1985

        # Load metadata
        self.metadata_file = self._load_metadata()

    def _load_metadata(self):
        raw_data_csv_file_link = f"https://perritos.myasustor.com:{self.port}/metadata.csv"
        return pd.read_csv(raw_data_csv_file_link, index_col=0)

    def _fetch_image(self, img_id):
        img_in_server_link = f"https://perritos.myasustor.com:{self.port}/data/img_id_{img_id}.jpg"
        response = requests.get(img_in_server_link)
        image = Image.open(BytesIO(response.content))
        return image

    def _process_image(self, img_id):
        image = self._fetch_image(img_id)
        if self.transform is not None:
            image = self.transform(image)
        else:
            image = tf.keras.preprocessing.image.img_to_array(image)
            image = image / 255.0  # Default normalization
        return image

    def _get_label(self, idx):
        return self.metadata_file["class"].iloc[idx]

    def _generator(self):
        if self.shuffle:
            img_indices = np.random.permutation(len(self.img_indx))
        else:
            img_indices = self.img_indx

        for idx in range(len(self.img_indx)):
            image = self._process_image(self.img_indx[idx])
            L, AB = image  # Unpack the transformed image
            if self.return_label:
                label = self._get_label(idx)
                yield (L, AB), label
            else:
                yield L, AB

    def get_dataset(self):
        # Dynamically infer the shapes of L and AB channels
        def _dynamic_output_signature():
            example_image = self._fetch_image(self.img_indx[0])
            example_transformed = self.transform(example_image)
            L, AB = example_transformed
            L_shape = tf.TensorSpec(shape=L.shape, dtype=tf.float32)
            AB_shape = tf.TensorSpec(shape=AB.shape, dtype=tf.float32)
            if self.return_label:
                return ((L_shape, AB_shape), tf.TensorSpec(shape=(), dtype=tf.int64))
            else:
                return (L_shape, AB_shape)

        output_signature = _dynamic_output_signature()

        dataset = tf.data.Dataset.from_generator(self._generator, output_signature=output_signature)
        dataset = dataset.batch(self.batch_size)
        return dataset

# Check info from the images

The data was initially created using the scripts `retrieve_data.ipynb` and stored in a private server for later (re)use.
In the metadata.csv file we get the information on original link, class and coordinates of each image.

NOTE: the following are links stored in a private server, jet they are still publically available.

In [8]:
is_large_dataset = True

if is_large_dataset:
    server_port = 1986 # Large dataset of ~10K images
else:
    server_port = 1985 # Large dataset of ~10K images
# server_port = 1985 # initial dataset of 3.6K images

raw_data_csv_file_link = f"https://perritos.myasustor.com:{server_port}/metadata.csv"


metadata_raw_df = pd.read_csv(raw_data_csv_file_link, index_col=0)
metadata_raw_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10008 entries, 0 to 10007
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   img_id      10008 non-null  int64  
 1   img_name    10008 non-null  object 
 2   latitude    10008 non-null  float64
 3   longitude   10008 non-null  float64
 4   zoom_level  10008 non-null  int64  
 5   class       10008 non-null  int64  
 6   link        10008 non-null  object 
dtypes: float64(2), int64(3), object(2)
memory usage: 625.5+ KB


# Split the Train, Valid and Test subsets.

We use the column `image_id` from the metadata as index of the images and then we perform standard shufling and splitting.

The final ratio for the train, validation and test dastasets are: 70, 29 and 1 % respectively

In [9]:
dataX, dataY = metadata_raw_df["img_id"].to_list(), metadata_raw_df["class"] .to_list()

rand_state = 9898
train_ratio = 0.89
validation_ratio = 0.10
test_ratio = 0.01

# train is now 75% of the entire data set
x_train, x_test, y_train, y_test = train_test_split(dataX, dataY, test_size=1 - train_ratio, stratify = dataY, random_state=rand_state)

# test is now 10% of the initial data set
# validation is now 15% of the initial data set
x_val, x_test, y_val, y_test = train_test_split(x_test, y_test, test_size=test_ratio/(test_ratio + validation_ratio), stratify=y_test, random_state=rand_state)

print(f"the size fo the train dataset is: {len(x_train)}.\nthe size fo the validation dataset is: {len(x_val)}.\nthe size fo the test dataset is: {len(x_test)}.")

the size fo the train dataset is: 8907.
the size fo the validation dataset is: 1000.
the size fo the test dataset is: 101.


# Dynamic Threshold

In [11]:
from einops import rearrange

def dynamic_threshold(img, percentile=0.8):
    s = torch.quantile(
        rearrange(img, 'b ... -> b (...)').abs(),
        percentile,
        dim=-1
    )
    # If threshold is less than 1, simply clamp values to [-1., 1.]
    s.clamp_(min=1.)
    s = right_pad_dims_to(img, s)
    # Clamp to +/- s and divide by s to bring values back to range [-1., 1.]
    img = img.clamp(-s, s) / s
    return img

# Configs

# Diffusion

In [13]:
from torch import optim
import torch.nn.functional as F
from pytorch_lightning import LightningModule

In [15]:
def linear_beta_schedule(timesteps, start=0.0001, end=0.02):
    return torch.linspace(start, end, timesteps)

def get_index_from_list(vals, t, x_shape):
    """
    Returns a specific index t of a passed list of values vals
    while considering the batch dimension.
    """
    batch_size = t.shape[0]
    vals = vals.to(t)
    out = vals.gather(-1, t.long())
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))

class GaussianDiffusion(LightningModule):
    def __init__(self, T, dynamic_threshold=False) -> None:
        super().__init__()
        self.betas = linear_beta_schedule(timesteps=T).to(self.device)
        self.alphas = 1. - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, axis=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / self.alphas)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - self.alphas_cumprod)
        self.posterior_variance = self.betas * (1. - self.alphas_cumprod_prev) / (1. - self.alphas_cumprod)
        self.dynamic_threshold=dynamic_threshold
    def forward_diff(self, x_0, t, T=300):
        """
        Takes an image and a timestep as input and noises the color channels to timestep t
        """
        l, ab = split_lab_channels(x_0)
        noise = torch.randn_like(ab)
        sqrt_alphas_cumprod_t = get_index_from_list(self.sqrt_alphas_cumprod, t, ab.shape).to(x_0)
        # print(f"sqrt_alphas_cumprod_t = {sqrt_alphas_cumprod_t}")
        sqrt_one_minus_alphas_cumprod_t = get_index_from_list(
            self.sqrt_one_minus_alphas_cumprod, t, ab.shape
        ).to(x_0)
        # print(f"sqrt_one_minus_alphas_cumprod_t = {sqrt_one_minus_alphas_cumprod_t}")
        # mean + variance
        ab_noised = sqrt_alphas_cumprod_t * ab \
        + sqrt_one_minus_alphas_cumprod_t * noise

        noised_img = torch.cat((l, ab_noised), dim=1)
        # lab_to_pil(noised_img).save("noised_img.png")
        # print(f"noise = {noise}")

        return(noised_img, noise)

    @torch.no_grad()
    def sample_timestep(self, model, encoder, x, t, cond=None, T=300, ema=None):
        x_l, x_ab = split_lab_channels(x)
        #gets the mean- and variance-derived variables for timestep t
        betas_t = get_index_from_list(self.betas.to(x), t, x.shape)
        sqrt_one_minus_alphas_cumprod_t = get_index_from_list(
            self.sqrt_one_minus_alphas_cumprod, t, x.shape
        )
        sqrt_recip_alphas_t = get_index_from_list(self.sqrt_recip_alphas, t, x.shape)
        posterior_variance_t = get_index_from_list(self.posterior_variance, t, x.shape)

        # Call model (current image - noise prediction)
        greyscale_emb = encoder(x_l)
        if ema is not None:
            with ema.average_parameters():
                pred = model(x, t, greyscale_emb)
        else:
            pred = model(x, t, greyscale_emb)
        beta_times_pred = betas_t * pred
        model_mean = sqrt_recip_alphas_t * (
            x_ab - beta_times_pred / sqrt_one_minus_alphas_cumprod_t
        )
        if t == 0:
            if self.dynamic_threshold:
                model_mean = dynamic_threshold(model_mean)
            return cat_lab(x_l, model_mean)
        else:
            noise = torch.randn_like(x_ab)
            ab_t_pred = model_mean + torch.sqrt(posterior_variance_t) * noise
            if self.dynamic_threshold:
                ab_t_pred = dynamic_threshold(ab_t_pred)
            return cat_lab(x_l, ab_t_pred)


In [16]:
d = GaussianDiffusion(T=300)

# Denoising

In [30]:
import math
import copy
from pathlib import Path
from random import random
from functools import partial
from collections import namedtuple
from multiprocessing import cpu_count

import torch
from torch import nn, einsum
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torch.optim import Adam
from torchvision import transforms as T, utils

from einops import rearrange, reduce
from einops.layers.torch import Rearrange

from PIL import Image
from tqdm.auto import tqdm

In [29]:
# constants

ModelPrediction =  namedtuple('ModelPrediction', ['pred_noise', 'pred_x_start'])

# helpers functions

def exists(x):
    return x is not None

def default(val, d):
    if exists(val):
        return val
    return d() if callable(d) else d

def identity(t, *args, **kwargs):
    return t

def cycle(dl):
    while True:
        for data in dl:
            yield data

def has_int_squareroot(num):
    return (math.sqrt(num) ** 2) == num

def num_to_groups(num, divisor):
    groups = num // divisor
    remainder = num % divisor
    arr = [divisor] * groups
    if remainder > 0:
        arr.append(remainder)
    return arr

def convert_image_to_fn(img_type, image):
    if image.mode != img_type:
        return image.convert(img_type)
    return image

# normalization functions

def normalize_to_neg_one_to_one(img):
    return img * 2 - 1

def unnormalize_to_zero_to_one(t):
    return (t + 1) * 0.5

# small helper modules

class Residual(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, *args, **kwargs):
        return self.fn(x, *args, **kwargs) + x

def Upsample(dim, dim_out = None, dropout = 0.2):
    if dropout: dropout /= 2
    return nn.Sequential(
        nn.Upsample(scale_factor = 2, mode = 'nearest'),
        nn.Dropout(p=dropout) if dropout > 0 else identity,
        nn.Conv2d(dim, default(dim_out, dim), 3, padding = 1)
    )

def Downsample(dim, dim_out = None, dropout = 0.5):
    return nn.Sequential(
        Rearrange('b c (h p1) (w p2) -> b (c p1 p2) h w', p1 = 2, p2 = 2),
        nn.Dropout(p=dropout) if dropout > 0 else identity,
        nn.Conv2d(dim * 4, default(dim_out, dim), 1)
    )

class WeightStandardizedConv2d(nn.Conv2d):
    """
    https://arxiv.org/abs/1903.10520
    weight standardization purportedly works synergistically with group normalization
    """
    def forward(self, x):
        eps = 1e-5 if x.dtype == torch.float32 else 1e-3

        weight = self.weight
        mean = reduce(weight, 'o ... -> o 1 1 1', 'mean')
        var = reduce(weight, 'o ... -> o 1 1 1', partial(torch.var, unbiased = False))
        normalized_weight = (weight - mean) * (var + eps).rsqrt()

        return F.conv2d(x, normalized_weight, self.bias, self.stride, self.padding, self.dilation, self.groups)

class LayerNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.g = nn.Parameter(torch.ones(1, dim, 1, 1))

    def forward(self, x):
        eps = 1e-5 if x.dtype == torch.float32 else 1e-3
        var = torch.var(x, dim = 1, unbiased = False, keepdim = True)
        mean = torch.mean(x, dim = 1, keepdim = True)
        return (x - mean) * (var + eps).rsqrt() * self.g

class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.fn = fn
        self.norm = LayerNorm(dim)

    def forward(self, x):
        x = self.norm(x)
        return self.fn(x)

# sinusoidal positional embeds

class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

class RandomOrLearnedSinusoidalPosEmb(nn.Module):
    """ following @crowsonkb 's lead with random (learned optional) sinusoidal pos emb """
    """ https://github.com/crowsonkb/v-diffusion-jax/blob/master/diffusion/models/danbooru_128.py#L8 """

    def __init__(self, dim, is_random = False):
        super().__init__()
        assert (dim % 2) == 0
        half_dim = dim // 2
        self.weights = nn.Parameter(torch.randn(half_dim), requires_grad = not is_random)

    def forward(self, x):
        x = rearrange(x, 'b -> b 1')
        freqs = x * rearrange(self.weights, 'd -> 1 d') * 2 * math.pi
        fouriered = torch.cat((freqs.sin(), freqs.cos()), dim = -1)
        fouriered = torch.cat((x, fouriered), dim = -1)
        return fouriered

# building block modules

class Block(nn.Module):
    def __init__(self, dim, dim_out, groups = 8):
        super().__init__()
        self.proj = WeightStandardizedConv2d(dim, dim_out, 3, padding = 1)
        self.norm = nn.GroupNorm(groups, dim_out)
        self.act = nn.SiLU()

    def forward(self, x, scale_shift = None):
        x = self.proj(x)
        x = self.norm(x)

        if exists(scale_shift):
            scale, shift = scale_shift
            x = x * (scale + 1) + shift

        x = self.act(x)
        return x

class ResnetBlock(nn.Module):
    def __init__(self, dim, dim_out, *, time_emb_dim = None, groups = 8):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, dim_out * 2)
        ) if exists(time_emb_dim) else None

        self.block1 = Block(dim, dim_out, groups = groups)
        self.block2 = Block(dim_out, dim_out, groups = groups)
        self.res_conv = nn.Conv2d(dim, dim_out, 1) if dim != dim_out else nn.Identity()

    def forward(self, x, time_emb = None):

        scale_shift = None
        if exists(self.mlp) and exists(time_emb):
            time_emb = self.mlp(time_emb)
            time_emb = rearrange(time_emb, 'b c -> b c 1 1')
            scale_shift = time_emb.chunk(2, dim = 1)

        h = self.block1(x, scale_shift = scale_shift)

        h = self.block2(h)

        return h + self.res_conv(x)

class LinearAttention(nn.Module):
    def __init__(self, dim, heads = 4, dim_head = 32):
        super().__init__()
        self.scale = dim_head ** -0.5
        self.heads = heads
        hidden_dim = dim_head * heads
        self.to_qkv = nn.Conv2d(dim, hidden_dim * 3, 1, bias = False)

        self.to_out = nn.Sequential(
            nn.Conv2d(hidden_dim, dim, 1),
            LayerNorm(dim)
        )

    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.to_qkv(x).chunk(3, dim = 1)
        q, k, v = map(lambda t: rearrange(t, 'b (h c) x y -> b h c (x y)', h = self.heads), qkv)

        q = q.softmax(dim = -2)
        k = k.softmax(dim = -1)

        q = q * self.scale
        v = v / (h * w)

        context = torch.einsum('b h d n, b h e n -> b h d e', k, v)

        out = torch.einsum('b h d e, b h d n -> b h e n', context, q)
        out = rearrange(out, 'b h c (x y) -> b (h c) x y', h = self.heads, x = h, y = w)
        return self.to_out(out)

class Attention(nn.Module):
    def __init__(self, dim, heads = 4, dim_head = 32):
        super().__init__()
        self.scale = dim_head ** -0.5
        self.heads = heads
        hidden_dim = dim_head * heads

        self.to_qkv = nn.Conv2d(dim, hidden_dim * 3, 1, bias = False)
        self.to_out = nn.Conv2d(hidden_dim, dim, 1)

    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.to_qkv(x).chunk(3, dim = 1)
        q, k, v = map(lambda t: rearrange(t, 'b (h c) x y -> b h c (x y)', h = self.heads), qkv)

        q = q * self.scale

        sim = einsum('b h d i, b h d j -> b h i j', q, k)
        attn = sim.softmax(dim = -1)
        out = einsum('b h i j, b h d j -> b h i d', attn, v)

        out = rearrange(out, 'b h (x y) d -> b (h d) x y', x = h, y = w)
        return self.to_out(out)


Model

In [22]:
class Unet(nn.Module):
    def __init__(
        self,
        dim,
        init_dim=None,
        dropout=0.,
        out_dim=None,
        dim_mults=(1, 2, 4, 8),
        channels=3,
        self_condition=False,
        condition=True,
        resnet_block_groups=8,
        learned_variance=False,
        learned_sinusoidal_cond=False,
        random_fourier_features=False,
        learned_sinusoidal_dim=16,
        **kwargs  # Additional kwargs can be accepted if needed
    ):
        super().__init__()

        # Use default values from unet_config if not provided in parameters
        unet_config = {
            'dim': dim,
            'init_dim': init_dim,
            'dropout': dropout,
            'out_dim': out_dim,
            'dim_mults': dim_mults,
            'channels': channels,
            'self_condition': self_condition,
            'condition': condition,
            'resnet_block_groups': resnet_block_groups,
            'learned_variance': learned_variance,
            'learned_sinusoidal_cond': learned_sinusoidal_cond,
            'random_fourier_features': random_fourier_features,
            'learned_sinusoidal_dim': learned_sinusoidal_dim
        }

        # Apply default values from unet_config using Python's **kwargs syntax
        unet_config = {**unet_config, **kwargs}

        # Now initialize the Unet with these configuration parameters
        self._init_unet(**unet_config)

    def _init_unet(
        self,
        dim,
        init_dim=None,
        dropout=0.,
        out_dim=None,
        dim_mults=(1, 2, 4, 8),
        channels=3,
        self_condition=False,
        condition=True,
        resnet_block_groups=8,
        learned_variance=False,
        learned_sinusoidal_cond=False,
        random_fourier_features=False,
        learned_sinusoidal_dim=16
    ):
        # Determine dimensions
        self.condition = condition
        self.channels = channels
        self.self_condition = self_condition
        self.dropout = dropout
        input_channels = channels * (2 if self_condition else 1)

        init_dim = self._default(init_dim, dim)
        self.init_conv = nn.Conv2d(input_channels, init_dim, 7, padding=3)

        dims = [init_dim, *map(lambda m: dim * m, dim_mults)]
        in_out = list(zip(dims[:-1], dims[1:]))

        block_klass = partial(ResnetBlock, groups=resnet_block_groups)

        # Time embeddings
        time_dim = dim * 4

        self.random_or_learned_sinusoidal_cond = learned_sinusoidal_cond or random_fourier_features

        if self.random_or_learned_sinusoidal_cond:
            sinu_pos_emb = RandomOrLearnedSinusoidalPosEmb(learned_sinusoidal_dim, random_fourier_features)
            fourier_dim = learned_sinusoidal_dim + 1
        else:
            sinu_pos_emb = SinusoidalPosEmb(dim)
            fourier_dim = dim

        self.time_mlp = nn.Sequential(
            sinu_pos_emb,
            nn.Linear(fourier_dim, time_dim),
            nn.GELU(),
            nn.Linear(time_dim, time_dim)
        )

        # Layers
        self.downs = nn.ModuleList([])
        self.ups = nn.ModuleList([])
        num_resolutions = len(in_out)

        for ind, (dim_in, dim_out) in enumerate(in_out):
            is_last = ind >= (num_resolutions - 1)

            self.downs.append(nn.ModuleList([
                block_klass(dim_in, dim_in, time_emb_dim=time_dim),
                block_klass(dim_in, dim_in, time_emb_dim=time_dim),
                Residual(PreNorm(dim_in, LinearAttention(dim_in))),
                Downsample(dim_in * 2, dim_out, dropout=self.dropout) if not is_last else nn.Conv2d(dim_in * 2, dim_out, 3, padding=1)
            ]))

        mid_dim = dims[-1]
        self.mid_block1 = block_klass(mid_dim, mid_dim, time_emb_dim=time_dim)
        self.mid_attn = Residual(PreNorm(mid_dim, Attention(mid_dim)))
        self.mid_block2 = block_klass(mid_dim, mid_dim, time_emb_dim=time_dim)

        for ind, (dim_in, dim_out) in enumerate(reversed(in_out)):
            is_last = ind == (len(in_out) - 1)

            self.ups.append(nn.ModuleList([
                block_klass(dim_out + dim_in, dim_out, time_emb_dim=time_dim),
                block_klass(dim_out + dim_in, dim_out, time_emb_dim=time_dim),
                Residual(PreNorm(dim_out, LinearAttention(dim_out))),
                Upsample(dim_out, dim_in, dropout=self.dropout) if not is_last else nn.Conv2d(dim_out, dim_in, 3, padding=1)
            ]))

        default_out_dim = channels * (1 if not learned_variance else 2)
        self.out_dim = self._default(out_dim, default_out_dim)

        self.final_res_block = block_klass(dim * 2, dim, time_emb_dim=time_dim)
        self.final_conv = nn.Conv2d(dim, self.out_dim, 1)

    def _default(self, value, default):
        return default if value is None else value

    def forward(self, x, time, greyscale_embs=None, x_self_cond=None):
        if self.self_condition:
            x_self_cond = self._default(x_self_cond, lambda: torch.zeros_like(x))
            x = torch.cat((x_self_cond, x), dim=1)

        x = self.init_conv(x)
        r = x.clone()

        t = self.time_mlp(time)

        h = []

        for i, (block1, block2, attn, downsample) in enumerate(self.downs):
            x = block1(x, t)
            h.append(x)

            x = block2(x, t)
            x = attn(x)
            h.append(x)
            x = torch.cat((x, greyscale_embs[i]), dim=1)
            x = downsample(x)

        x = self.mid_block1(x, t)
        x = self.mid_attn(x)
        x = self.mid_block2(x, t)

        for block1, block2, attn, upsample in self.ups:
            x = torch.cat((x, h.pop()), dim=1)
            x = block1(x, t)

            x = torch.cat((x, h.pop()), dim=1)
            x = block2(x, t)
            x = attn(x)

            x = upsample(x)

        x = torch.cat((x, r), dim=1)

        x = self.final_res_block(x, t)
        return self.final_conv(x)

Encoder

In [25]:
import torch
import torch.nn as nn
from functools import partial

class Encoder(nn.Module):
    def __init__(self, channels=1, dropout=0.3, self_condition=False, out_dim=2, dim=128, dim_mults=[1, 2, 3, 3]):
        super().__init__()

        self.dropout = dropout
        self.channels = channels
        self.self_condition = self_condition
        input_channels = channels * (2 if self_condition else 1)

        init_dim = dim
        self.init_conv = nn.Conv2d(input_channels, init_dim, 7, padding=3)

        dims = [init_dim, *map(lambda m: dim * m, dim_mults)]
        in_out = list(zip(dims[:-1], dims[1:]))

        block_klass = partial(ResnetBlock, groups=8)

        # Time embeddings
        time_dim = dim * 4

        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(dim),
            nn.Linear(dim, time_dim),
            nn.GELU(),
            nn.Linear(time_dim, time_dim)
        )

        self.downs = nn.ModuleList([])
        num_resolutions = len(in_out)

        for ind, (dim_in, dim_out) in enumerate(in_out):
            is_last = ind >= (num_resolutions - 1)

            self.downs.append(nn.ModuleList([
                block_klass(dim_in, dim_in, time_emb_dim=time_dim),
                block_klass(dim_in, dim_in, time_emb_dim=time_dim),
                Residual(PreNorm(dim_in, LinearAttention(dim_in))),
                nn.Conv2d(dim_in, dim_out, 3, padding=1) if is_last else Downsample(dim_in, dim_out, dropout=self.dropout)
            ]))

    def forward(self, x):
        intermediates = []
        x = self.init_conv(x)

        t = None  # Replace with your time variable

        for block1, block2, attn, downsample in self.downs:
            x = block1(x, t)
            x = block2(x, t)
            x = attn(x)
            intermediates.append(x)
            x = downsample(x)

        return intermediates

In [12]:
encoder_config = {
    'channels': 1,
    'dropout': 0.3,
    'self_condition': False,
    'out_dim': 2,
    'dim': 128,
    'dim_mults': [1, 2, 3, 3]
}

unet_config = {
    'channels': 3,
    'dropout': 0.3,
    'self_condition': False,
    'out_dim': 2,
    'dim': 128,
    'condition': True,
    'dim_mults': [1, 2, 3, 3]
}

In [26]:
# Instantiate Unet
unet = Unet(**unet_config)

# Print the instantiated Unet object
print(unet)

Unet(
  (init_conv): Conv2d(3, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3))
  (time_mlp): Sequential(
    (0): SinusoidalPosEmb()
    (1): Linear(in_features=128, out_features=512, bias=True)
    (2): GELU(approximate='none')
    (3): Linear(in_features=512, out_features=512, bias=True)
  )
  (downs): ModuleList(
    (0): ModuleList(
      (0-1): 2 x ResnetBlock(
        (mlp): Sequential(
          (0): SiLU()
          (1): Linear(in_features=512, out_features=256, bias=True)
        )
        (block1): Block(
          (proj): WeightStandardizedConv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (norm): GroupNorm(8, 128, eps=1e-05, affine=True)
          (act): SiLU()
        )
        (block2): Block(
          (proj): WeightStandardizedConv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (norm): GroupNorm(8, 128, eps=1e-05, affine=True)
          (act): SiLU()
        )
        (res_conv): Identity()
      )
      (2): 

In [24]:
# Instantiate Encoder
encoder = Encoder(**encoder_config)

# Example of using the Encoder
input_tensor = torch.randn(1, 1, 64, 64)  # Example input tensor
output = encoder(input_tensor)

# Print output shape
for i, intermediate in enumerate(output):
    print(f"Intermediate {i + 1} shape:", intermediate.shape)

Intermediate 1 shape: torch.Size([1, 128, 64, 64])
Intermediate 2 shape: torch.Size([1, 128, 32, 32])
Intermediate 3 shape: torch.Size([1, 256, 16, 16])
Intermediate 4 shape: torch.Size([1, 384, 8, 8])


# Model

In [19]:
from tqdm import tqdm
import torchvision
import pytorch_lightning as pl
from matplotlib import pyplot as plt


In [20]:
class ColorDiffusion(pl.LightningModule):
    def __init__(self,
                 unet,
                 train_dl,
                 val_dl,
                 encoder,
                 loss_fn="l2",
                 T=300,
                 lr=1e-4,
                 batch_size=12,
                 sample=True,
                 should_log=True,
                 using_cond=False,
                 display_every=None,
                 dynamic_threshold=False,
                 use_ema=True,
                 **kwargs):
        super().__init__()
        self.unet = unet.to(self.device)
        self.T = T
        self.lr = lr
        self.using_cond = using_cond
        self.sample = sample
        self.should_log = should_log
        self.encoder = encoder
        self.display_every = display_every
        self.val_dl = val_dl
        self.train_dl = train_dl
        if loss_fn == "l1":
            self.loss_fn = torch.nn.functional.l1_loss
        else:
            self.loss_fn = torch.nn.functional.mse_loss

        self.ema = ExponentialMovingAverage(self.unet.parameters(),
                                            decay=0.9999)
        self.ema.to(self.device)
        self.diffusion = GaussianDiffusion(T,
                                           dynamic_threshold=dynamic_threshold)
        if sample is True and display_every is None:
            display_every = 1000
        self.save_hyperparameters(ignore=['unet'])

    def forward(self, x_noised, t, x_l):
        """
        Performs one denoising step on batch of noised inputs
        Unet is conditioned on timestep and features extracted from greyscale channel
        """
        cond = self.encoder(x_l)
        noise_pred = self.unet(x_noised, t, greyscale_embs=cond)
        return noise_pred

    def get_batch_pred(self, x_0, x_l):
        """
        Samples a timestep from range [0, T]
        Adds noise to images x_0 to get x_t (x_0 with color channels noised)
        Returns:
        - The model's prediction of the noise,
        - The real noise applied to the color channels by the forward diffusion process
        """
        t = torch.randint(0, self.T, (x_0.shape[0],)).to(x_0)
        x_noised, noise = self.diffusion.forward_diff(x_0, t, T=self.T)
        return (self(x_noised, t, x_l), noise)

    def get_losses(self, noise_pred, noise, x_l):
        diff_loss = self.loss_fn(noise_pred, noise)
        return {"total loss": diff_loss}

    def training_step(self, x_0, batch_idx):
        x_l, _ = split_lab_channels(x_0)
        noise_pred, noise = self.get_batch_pred(x_0, x_l)
        losses = self.get_losses(noise_pred, noise, x_l)
        self.log_dict(losses, on_step=True)
        if self.sample and batch_idx and batch_idx % self.display_every == 0 and self.global_step > 1:
            self.test_step(x_0)
        return losses["total loss"]

    def validation_step(self, batch, batch_idx):
        x_l, _ = split_lab_channels(batch)
        noise_pred, noise = self.get_batch_pred(batch, x_l)
        losses = self.get_losses(noise_pred, noise, x_l)
        if self.should_log:
            self.log("val_loss", losses["total loss"])
        if self.sample and batch_idx and batch_idx % self.display_every == 0:
            self.sample_plot_image(batch)
        return losses["total loss"]

    @torch.inference_mode()
    def test_step(self, batch, *args, **kwargs):
        x = next(iter(self.val_dl)).to(batch)
        self.sample_plot_image(x)
        self.sample_plot_image(x, use_ema=True)

    def configure_optimizers(self):
        learnable_params = list(self.unet.parameters()) \
                            + list(self.encoder.parameters())
        global_optim = torch.optim.AdamW(learnable_params,
                                         lr=self.lr,
                                         weight_decay=28e-3)
        return global_optim

    def log_img(self, image, caption="diff samples", use_ema=False):
        rgb_imgs = lab_to_rgb(*split_lab_channels(image))
        if use_ema:
            self.logger.log_image("EMA samples", [rgb_imgs])
        else:
            self.logger.log_image("samples", [rgb_imgs])

    def on_before_zero_grad(self, *args, **kwargs):
        self.ema.update()

    @torch.inference_mode()
    def sample_loop(self, x_l, prog=False, use_ema=False, save_all=False):
        """
        Noises color channels to timestep T, then denoises the color channels
        to t=0 to get the colorized image.
        Returns an array containing the noised image,
        intermediate images in the denoising process, and the final image
        """
        ema = self.ema if use_ema else None
        images = []
        num_images = 13
        img_size = x_l.shape[-1]
        stepsize = self.T // num_images

        # Initialize image with random noise in color channels
        x_ab = torch.randn((x_l.shape[0], 2, img_size, img_size)).to(x_l)
        img = torch.cat((x_l, x_ab), dim=1)

        counter = range(0, self.T)[::-1]
        if prog:
            counter = tqdm(counter)
        for i in counter:
            t = torch.full((1,), i, dtype=torch.long).to(img)

            img = self.diffusion.sample_timestep(self.unet,
                                                 self.encoder,
                                                 img,
                                                 t,
                                                 T=self.T,
                                                 cond=x_l,
                                                 ema=ema)
            if i % stepsize == 0:
                images += img.unsqueeze(0)
            if save_all and i % 2 == 0:
                pil_img = lab_to_pil(img)
                pil_img.save(f"./visualization/denoising/{i:04d}.png")
        return images

    @torch.inference_mode()
    def sample_plot_image(self, x_0, show=True, prog=False,
                          use_ema=False, log=True, save_all=False):
        """
        Denoises a single image and displays a grid showing:
        - ground truth image
        - intermediate denoised outputs
        - the final denoised image
        """
        print("Sampling image")
        ground_truth_images = []
        if x_0.shape[1] == 3:
            x_l, _ = split_lab_channels(x_0)
            ground_truth_images.append(x_0[:1])
        else:
            x_l = x_0
        x_l = x_l[:1]
        greyscale = torch.cat((x_l, *[torch.zeros_like(x_l)] * 2), dim=1)
        ground_truth_images += greyscale.unsqueeze(0)
        if len(x_l.shape) == 3:
            x_l = x_l.unsqueeze(0)
        images = ground_truth_images + self.sample_loop(x_l,
                                                        prog=prog,
                                                        use_ema=use_ema,
                                                        save_all=save_all)
        grid = torchvision.utils.make_grid(torch.cat(images), dim=0).to(x_l)
        if show:
            show_lab_image(grid.unsqueeze(0), log=self.should_log)
            plt.show()
        if self.should_log and log:
            self.log_img(grid.unsqueeze(0), use_ema=use_ema)
        return lab_to_rgb(*split_lab_channels(images[-1]))

In [27]:
import torch
import pytorch_lightning as pl
import torchvision
import matplotlib.pyplot as plt
from tqdm import tqdm
from functools import partial

# Assuming you have defined the necessary classes like GaussianDiffusion, ExponentialMovingAverage, and split_lab_channels

class ColorDiffusion(pl.LightningModule):
    def __init__(self,
                 unet,
                 train_dl,
                 val_dl,
                 encoder,
                 loss_fn="l2",
                 T=300,
                 lr=1e-4,
                 batch_size=12,
                 sample=True,
                 should_log=True,
                 using_cond=False,
                 display_every=None,
                 dynamic_threshold=False,
                 use_ema=True,
                 **kwargs):
        super().__init__()
        self.unet = unet.to(self.device)
        self.T = T
        self.lr = lr
        self.using_cond = using_cond
        self.sample = sample
        self.should_log = should_log
        self.encoder = encoder
        self.display_every = display_every
        self.val_dl = val_dl
        self.train_dl = train_dl
        if loss_fn == "l1":
            self.loss_fn = torch.nn.functional.l1_loss
        else:
            self.loss_fn = torch.nn.functional.mse_loss

        self.ema = ExponentialMovingAverage(self.unet.parameters(),
                                            decay=0.9999)
        self.ema.to(self.device)
        self.diffusion = GaussianDiffusion(T,
                                           dynamic_threshold=dynamic_threshold)
        if sample is True and display_every is None:
            display_every = 1000
        self.save_hyperparameters(ignore=['unet'])

    def forward(self, x_noised, t, x_l):
        """
        Performs one denoising step on batch of noised inputs
        Unet is conditioned on timestep and features extracted from greyscale channel
        """
        cond = self.encoder(x_l)
        noise_pred = self.unet(x_noised, t, greyscale_embs=cond)
        return noise_pred

    def get_batch_pred(self, x_0, x_l):
        """
        Samples a timestep from range [0, T]
        Adds noise to images x_0 to get x_t (x_0 with color channels noised)
        Returns:
        - The model's prediction of the noise,
        - The real noise applied to the color channels by the forward diffusion process
        """
        t = torch.randint(0, self.T, (x_0.shape[0],)).to(x_0)
        x_noised, noise = self.diffusion.forward_diff(x_0, t, T=self.T)
        return (self(x_noised, t, x_l), noise)

    def get_losses(self, noise_pred, noise, x_l):
        diff_loss = self.loss_fn(noise_pred, noise)
        return {"total loss": diff_loss}

    def training_step(self, x_0, batch_idx):
        x_l, _ = split_lab_channels(x_0)
        noise_pred, noise = self.get_batch_pred(x_0, x_l)
        losses = self.get_losses(noise_pred, noise, x_l)
        self.log_dict(losses, on_step=True)
        if self.sample and batch_idx and batch_idx % self.display_every == 0 and self.global_step > 1:
            self.test_step(x_0)
        return losses["total loss"]

    def validation_step(self, batch, batch_idx):
        x_l, _ = split_lab_channels(batch)
        noise_pred, noise = self.get_batch_pred(batch, x_l)
        losses = self.get_losses(noise_pred, noise, x_l)
        if self.should_log:
            self.log("val_loss", losses["total loss"])
        if self.sample and batch_idx and batch_idx % self.display_every == 0:
            self.sample_plot_image(batch)
        return losses["total loss"]

    @torch.inference_mode()
    def test_step(self, batch, *args, **kwargs):
        x = next(iter(self.val_dl)).to(batch)
        self.sample_plot_image(x)
        self.sample_plot_image(x, use_ema=True)

    def configure_optimizers(self):
        learnable_params = list(self.unet.parameters()) \
                            + list(self.encoder.parameters())
        global_optim = torch.optim.AdamW(learnable_params,
                                         lr=self.lr,
                                         weight_decay=28e-3)
        return global_optim

    def log_img(self, image, caption="diff samples", use_ema=False):
        rgb_imgs = lab_to_rgb(*split_lab_channels(image))
        if use_ema:
            self.logger.log_image("EMA samples", [rgb_imgs])
        else:
            self.logger.log_image("samples", [rgb_imgs])

    def on_before_zero_grad(self, *args, **kwargs):
        self.ema.update()

    @torch.inference_mode()
    def sample_loop(self, x_l, prog=False, use_ema=False, save_all=False):
        """
        Noises color channels to timestep T, then denoises the color channels
        to t=0 to get the colorized image.
        Returns an array containing the noised image,
        intermediate images in the denoising process, and the final image
        """
        ema = self.ema if use_ema else None
        images = []
        num_images = 13
        img_size = x_l.shape[-1]
        stepsize = self.T // num_images

        # Initialize image with random noise in color channels
        x_ab = torch.randn((x_l.shape[0], 2, img_size, img_size)).to(x_l)
        img = torch.cat((x_l, x_ab), dim=1)

        counter = range(0, self.T)[::-1]
        if prog:
            counter = tqdm(counter)
        for i in counter:
            t = torch.full((1,), i, dtype=torch.long).to(img)

            img = self.diffusion.sample_timestep(self.unet,
                                                 self.encoder,
                                                 img,
                                                 t,
                                                 T=self.T,
                                                 cond=x_l,
                                                 ema=ema)
            if i % stepsize == 0:
                images += img.unsqueeze(0)
            if save_all and i % 2 == 0:
                pil_img = lab_to_pil(img)
                pil_img.save(f"./visualization/denoising/{i:04d}.png")
        return images

    @torch.inference_mode()
    def sample_plot_image(self, x_0, show=True, prog=False,
                          use_ema=False, log=True, save_all=False):
        """
        Denoises a single image and displays a grid showing:
        - ground truth image
        - intermediate denoised outputs
        - the final denoised image
        """
        print("Sampling image")
        ground_truth_images = []
        if x_0.shape[1] == 3:
            x_l, _ = split_lab_channels(x_0)
            ground_truth_images.append(x_0[:1])
        else:
            x_l = x_0
        x_l = x_l[:1]
        greyscale = torch.cat((x_l, *[torch.zeros_like(x_l)] * 2), dim=1)
        ground_truth_images += greyscale.unsqueeze(0)
        if len(x_l.shape) == 3:
            x_l = x_l.unsqueeze(0)
        images = ground_truth_images + self.sample_loop(x_l,
                                                        prog=prog,
                                                        use_ema=use_ema,
                                                        save_all=save_all)
        grid = torchvision.utils.make_grid(torch.cat(images), dim=0).to(x_l)
        if show:
            show_lab_image(grid.unsqueeze(0), log=self.should_log)
            plt.show()
        if self.should_log and log:
            self.log_img(grid.unsqueeze(0), use_ema=use_ema)
        return lab_to_rgb(*split_lab_channels(images[-1]))

In [28]:
# Define colordiff_config with specific parameters
colordiff_config = {
    'device': 'auto',
    'pin_memory': True,
    'T': 350,
    'lr': 1.0e-06,
    'loss_fn': 'l2',
    'batch_size': 36,
    'accumulate_grad_batches': 2,
    'img_size': 64,
    'sample': True,
    'should_log': True,
    'epochs': 14,
    'using_cond': True,
    'display_every': 350,
    'dynamic_threshold': False,
    'train_autoenc': False,
    'enc_loss_coeff': 1.1
}

# Instantiate Unet using unet_config
unet = Unet(**unet_config)

# Instantiate Encoder using encoder_config
encoder = Encoder(**encoder_config)

# Instantiate ColorDiffusion using colordiff_config, Unet, and Encoder
color_diffusion = ColorDiffusion(unet=unet,
                                 train_dl=train_dl,
                                 val_dl=val_dl,
                                 encoder=encoder,
                                 **colordiff_config)

# Print the instantiated ColorDiffusion object
print(color_diffusion)

NameError: name 'train_dl' is not defined